In [ ]:
# Installing dependencies
!pip install datasets transformers torchaudio librosa evaluate jiwer gradio soundfile

In [ ]:
# Importing necessary libraries
import torch
import torchaudio
import librosa
import numpy as np

from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Trainer,
    TrainingArguments
)
import evaluate

In [ ]:
# Loading the dataset
dataset = load_dataset("librispeech_asr", "clean", split="train.100[:1%]")

dataset

In [ ]:
# Audio Preprocessing
def preprocess_audio(batch):
    audio = batch["audio"]
    waveform, sr = torchaudio.load(audio["path"])
    waveform = torchaudio.functional.resample(waveform, sr, 16000)
    batch["audio"] = waveform.squeeze().numpy()
    return batch

dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
# Train Test Split
dataset = dataset.train_test_split(test_size=0.1)

In [ ]:
# Load Whisper model
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
# Tokenization
def prepare_dataset(batch):
    audio = batch["audio"]

    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]

    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

dataset = dataset.map(prepare_dataset)

In [ ]:
# WER Metric
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Training Arguments
training_args = TrainingArguments(
    output_dir="./stt_model",
    per_device_train_batch_size=8,
    evaluation_strategy="steps",
    num_train_epochs=2,
    save_steps=500,
    logging_steps=100,
    learning_rate=1e-5,
    fp16=True,
)

In [ ]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=processor.feature_extractor,
    compute_metrics=compute_metrics,
)

In [ ]:
# Train
trainer.train()

In [ ]:
# Saving the model
trainer.save_model("stt_model")
processor.save_pretrained("stt_model")